In [1]:
/*
  This notebook is intended for the app hosted at 
    https://mooreolith.github.io/notebook
  and can be url loaded via the following link:
    https://mooreolith.github.io/notebook/?url=./notebooks/golem-population.ipynb
  which opens the app and loads the notebook at the relative url.

  To run the visualization, scroll to the bottom of the page, and click
    Run Cell
  
  Enjoy!
*/
var vegaModule = await import("https://cdn.jsdelivr.net/npm/vega@5");
var vegaLiteModule = await import("https://cdn.jsdelivr.net/npm/vega-lite@5");
var vegaEmbedModule = await import("https://cdn.jsdelivr.net/npm/vega-embed@6");

/*
  Constants
*/
var Constants = {
  speechBonus: 1.5,
  listenBonus: 1.5,
  communications: 100
}

function choice(l=[]){
  return l[Math.floor(Math.random() * l.length)];
}

function mean(l=[]){
  var sum = 0.0;
  for(var i=0; i<l.length; i++){
    sum += l[i];
  }
  return sum/l.length;
}

/*
  Golem

  These golems consist of a health, starting as a random number between 0 
  and 1, and a health, starting at 0. 
  
  Each round, a pair of golems is chosen to speak. They speak to each other,
  modelling what they say on themselves, before speaking to another golem, 
  changing their health, for better or worse. 

  Each round, each golem ages by 1. At the age of 100, a golem dies. 
*/
function Golem(){
  this.health = Math.random();
  this.age = 0;
}

Golem.prototype.speak = function(){
  var impact = (choice([-1, 1]) * Math.random() * Constants.speechBonus);
  this.health += impact;
  return impact;
}

Golem.prototype.listenTo = function(speech){
  var impact = (Math.random() * speech) + Constants.listenBonus;
  this.health *= impact;
}

/*
  Golem Society

  The golem society administers the rounds, and records its state for 
  visualization in the VegaLite Visualization. 
*/
function GolemSociety(n){
  this.golems = [];
  for(var i=0; i<n; i++){
    this.golems.push(new Golem());
  }
}

GolemSociety.prototype.historize = function(fact){
  if(this.history === undefined){
    this.history = {};
  }
  
  for(var attribute of Object.keys(fact)){
    if(this.history[attribute] === undefined){
      this.history[attribute] = [];
    }
    
    this.history[attribute].push(fact[attribute]);
  }
}

GolemSociety.prototype.round = function(){
  var speaker = choice(this.golems);
  var listener = choice(this.golems);

  var speech = speaker.speak();
  listener.listenTo(speech);

  if(listener.health > 1.0){
    this.golems.push(new Golem());
    listener.health *= 0.5;
  }

  if(listener.health < 0.0){
    this.golems.splice(this.golems.indexOf(listener), 1);
  }

  this.golems.forEach((golem, i) => {
    golem.age += 1;
    golem.age > 100 ? this.golems.splice(i, 1) : 0;
  });

  var healths = this.golems.map(g => g.health);
  var ages = this.golems.map(g => g.age);

  var meanHealth = mean(healths);
  var meanAge = mean(ages);

  var minHealth = Math.min(...healths);
  var maxHealth = Math.max(...healths);

  var minAge = Math.min(...ages);
  var maxAge = Math.max(...ages);

  var labels = {
    mean_health_label: "Mean Health",
    mean_age_label: "Mean Age"
  }
  
  var data = {
    round      : this.history.round.length,
    count      : this.golems.length,
    meanHealth: meanHealth,
    meanAge   : meanAge,
    ...labels
  }

  this.historize(data);
  return data;
}

var society = new GolemSociety(10)

/*
  The VegaLite, VegaEmbed Visualization of data described above (and 
  collected in the generator function below). 

  Citation: 
  @article{2017-vega-lite,
    doi = {10.1109/tvcg.2016.2599030},
    year = {2017},
    author = {Arvind Satyanarayan and Dominik Moritz and Kanit Wongsuphasawat and Jeffrey Heer},
    title = {Vega-Lite: A Grammar of Interactive Graphics},
    journal = {{IEEE} Transactions on Visualization \& Computer Graphics (Proc. InfoVis)},
    url = {http://idl.cs.washington.edu/papers/vega-lite},
  }
*/
vegaEmbed(output, {
  $schema: "https://vega.github.io/schema/vega-lite/v5.json",
  title: "Golem Health, Age & Population Count",
  data: {name: 'table'},
  width: 550,
  height: 350,
  autosize: {
    resize: true,
    type: "fit",
    contains: "padding"
  },
  layer: [
    {
      mark: "line",
      encoding: {
        x: {
          title: "Iteration", 
          field: "round", 
          type: "ordinal"
        },
        y: {
          title: "Mean Health", 
          field: "meanHealth", 
          type: "quantitative", 
          scale: {domain: [0, 1]},
          axis: {orient: "right", offset: 20, labelColor: "orange"}
        },
        color: {value: "orange"}
      }
    },
    {
      mark: "line",
      encoding: {
        x: {
          field: "round",
          type: "ordinal"
        },
        y: {
          title: "Mean Age",
          field: "meanAge", 
          type: "quantitative",
          scale: {domain: [0, 100]},
          axis: {orient: "right", offset: 80, labelColor: "cornflowerblue"}
        },
        color: {value: "cornflowerblue"}
      }
    },
    {
      mark: "line",
      encoding: {
        x: {
          field: "round",
          type: "ordinal"
        },
        y: {
          title: "Population Count",
          field: "count",
          type: "quantitative",
          scale: {domain: [0, 100]},
          axis: {orient: "right", offset: 140, labelColor: "red"}
        },
        color: {value: "red"}
      }
    }
  ],
  resolve: {scale: {y: "independent"}}
}).then(function(res) {
  function newGenerator(){
    var counter = -1;
    var previousY = [0];
  
    return function(){
      counter++;
      
      var newVals = previousY.map(() => society.round());
      
      previousY = newVals.map(function(v){
        return v['mean_health'];
      });
      
      return newVals;
    };
  }

  var valueGenerator = newGenerator();
  var minimumX = -25;
  window.setInterval(function () {
    minimumX++;
    var changeSet = vega
      .changeset()
      .insert(valueGenerator())
      .remove((t) => t['round'] < minimumX);
    res.view.change('table', changeSet).run();
  }, 50);
});

society.historize({round: 0, mean_health: 0, mean_age: 0, count: 0});




  
  
  
Save as SVGSave as PNGView SourceView Compiled VegaOpen in Vega Editor